In [2]:
!pip install numpy==1.24.4 pandas==2.0.3

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# -*- coding: utf-8 -*-
"""
-------------------------------------------------
   File Name：     logo_parse
   Author :       huangkai
   date：          2025/6/24
-------------------------------------------------
"""
import pandas as pd


def OperationLog(log_data):
    # 假设日志数据存储在CSV文件中
    # 提取主帐号名称、操作时间和操作内容
    log_data = log_data[['主帐号名称', '操作时间', '操作内容']]

    # 确保操作时间格式正确
    log_data['操作时间'] = pd.to_datetime(log_data['操作时间'], errors='coerce')
    log_data = log_data.dropna(subset=['操作时间'])

    # 按主帐号名称分类统计
    account_counts = log_data['主帐号名称'].value_counts()

    log_data['日期'] = log_data['操作时间'].dt.date
    daily_counts = log_data.groupby('日期').size()

    # 按操作内容分类统计
    operation_counts = log_data['操作内容'].value_counts()

    # 多维度组合分析：按主账号和操作时间分组
    account_daily_counts = log_data.groupby(['主帐号名称', '日期']).size().unstack(fill_value=0)

    error_time = log_data[((log_data['操作时间'].dt.hour >= 0) & (log_data['操作时间'].dt.hour < 6))]

    error_login_daily_counts = error_time.groupby(['主帐号名称', '日期']).size().unstack(fill_value=0)

    error_login_counts = error_time['主帐号名称'].value_counts()

    return account_counts, daily_counts, operation_counts, account_daily_counts, error_login_counts, error_login_daily_counts


def UseLog(log_data):
    # 提取登录ID、操作内容和访问时间
    log_data = log_data[['登录ID', '操作内容', '访问时间']]

    # 确保访问时间格式正确
    log_data['访问时间'] = pd.to_datetime(log_data['访问时间'], format='%Y%m%d%H%M%S', errors='coerce')
    log_data = log_data.dropna(subset=['访问时间'])

    # 按登录ID分类统计
    login_id_counts = log_data['登录ID'].value_counts()

    # 按操作内容分类统计
    operation_counts = log_data['操作内容'].value_counts()

    # 按访问时间分类统计
    log_data['日期'] = log_data['访问时间'].dt.date
    daily_counts = log_data.groupby('日期').size()

    # 多维度组合分析：按登录ID和操作内容分组
    login_operation_counts = log_data.groupby(['登录ID', '操作内容']).size().unstack(fill_value=0)

    # 多维度组合分析：按登录ID和日期分组
    login_daily_counts = log_data.groupby(['登录ID', '日期']).size().unstack(fill_value=0)

    error_time = log_data[(log_data['访问时间'].dt.hour >= 0) & (log_data['访问时间'].dt.hour < 6)]

    error_login_daily_counts = error_time.groupby(['登录ID', '日期']).size().unstack(fill_value=0)

    error_login_counts = error_time['登录ID'].value_counts()

    return login_id_counts, operation_counts, daily_counts, login_operation_counts, login_daily_counts, error_login_counts, error_login_daily_counts


def EnterLog(log_data):
    # 提取登录ID、登录时间、登出时间和客户端浏览器
    log_data = log_data[['登录ID', '登录时间', '登出时间', '客户端浏览器']]

    # 确保登录时间和登出时间格式正确
    log_data['登录时间'] = pd.to_datetime(log_data['登录时间'], format='%Y%m%d%H%M%S', errors='coerce')
    log_data['登出时间'] = pd.to_datetime(log_data['登出时间'], format='%Y%m%d%H%M%S', errors='coerce')
    log_data = log_data.dropna(subset=['登录时间', '登出时间'])
    # 数据可视化：登录ID登录频率图
    login_counts = log_data['登录ID'].value_counts()[:30]

    # 按登录时间分类统计
    log_data['日期'] = log_data['登录时间'].dt.date
    daily_counts = log_data.groupby('日期').size()

    # 每个登录ID在一天之内的登录次数排序
    log_data = log_data[['登录ID', '登录时间']]

    # 确保登录时间格式正确
    log_data['登录时间'] = pd.to_datetime(log_data['登录时间'], format='%Y%m%d%H%M%S', errors='coerce')
    log_data = log_data.dropna(subset=['登录时间'])

    # 提取日期部分
    log_data['日期'] = log_data['登录时间'].dt.date

    # 按登录ID和日期分组，统计每个ID在每一天的登录次数
    daily_login_counts = log_data.groupby(['登录ID', '日期']).size().reset_index(name='登录次数')

    # 按登录次数降序排序
    sorted_daily_login_counts = daily_login_counts.sort_values(by='登录次数', ascending=False)
    pivot_table = sorted_daily_login_counts[:30].pivot(index='登录ID', columns='日期', values='登录次数')

    pivot_table.fillna(0, inplace=True)

    error_time = log_data[(log_data['登录时间'].dt.hour >= 0) & (log_data['登录时间'].dt.hour < 6)]
    error_daily_login_counts = error_time.groupby(['登录ID', '日期']).size().reset_index(name='登录次数')
    error_sorted_daily_login_counts = error_daily_login_counts.sort_values(by='登录次数', ascending=False)
    error_pivot_table = error_sorted_daily_login_counts[:30].pivot(index='登录ID', columns='日期',
                                                                   values='登录次数').fillna(0)

    error_login_counts = error_time['登录ID'].value_counts()

    return login_counts, daily_counts, pivot_table, error_login_counts, error_pivot_table

# if __name__ == '__main__':
#     data =pd.read_excel(r"D:\code\langchain-chat-zt\tests\logparse\操作日志查验：应用侧近半年日志(四川).xlsx")
#     print(UseLog(data))



In [24]:
log_data =pd.read_excel(r"1-1日志：使用日志-下载记录-云数据安全防护能力.xlsx")


In [25]:
# 提取登录ID、操作内容和访问时间
log_data = log_data[['登录ID', '操作内容', '访问时间']]

# 确保访问时间格式正确
log_data['访问时间'] = pd.to_datetime(log_data['访问时间'], format='%Y%m%d%H%M%S', errors='coerce')

In [26]:
# log_data.to_excel("1-1日志：使用日志-下载记录-云数据安全防护能力.xlsx")

In [27]:

log_data = log_data.dropna(subset=['访问时间'])

# 按登录ID分类统计
login_id_counts = log_data['登录ID'].value_counts()

# 按操作内容分类统计
operation_counts = log_data['操作内容'].value_counts()

# 按访问时间分类统计
log_data['日期'] = log_data['访问时间'].dt.date
daily_counts = log_data.groupby('日期').size()
# 多维度组合分析：按登录ID和操作内容分组
login_operation_counts = log_data.groupby(['登录ID', '操作内容']).size().unstack(fill_value=0).to_dict()
all_items = []
for date, inner_dict in login_operation_counts.items():
    for key, value in inner_dict.items():
        all_items.append((date, key, value))

# 按值降序排序
all_items.sort(key=lambda x: x[2], reverse=True)

# 获取前20个最大值
login_operation_counts = all_items[:50]

# 多维度组合分析：按登录ID和日期分组
login_daily_counts = log_data.groupby(['登录ID', '日期']).size().unstack(fill_value=0)
all_items = []
for date, inner_dict in login_daily_counts.items():
    for key, value in inner_dict.items():
        all_items.append((date, key, value))

# 按值降序排序
all_items.sort(key=lambda x: x[2], reverse=True)

# 获取前20个最大值
login_daily_counts = all_items[:50]



#+++++++++++++++++++++++++++++
error_time = log_data[(log_data['访问时间'].dt.hour >= 0) & (log_data['访问时间'].dt.hour < 6)]
#+++++++++++++++++++++++++++++
error_login_daily_counts = error_time.groupby(['登录ID', '日期']).size().unstack(fill_value=0).to_dict()
error_login_daily_counts = {
    date: {k: v for k, v in inner_dict.items() if v != 0}
    for date, inner_dict in error_login_daily_counts.items()
}
#++++++++++++++++++++++++
error_login_counts = error_time['登录ID'].value_counts()

In [28]:
len(str(login_id_counts.to_dict())),len(str(operation_counts.to_dict())),len(str(daily_counts.to_dict())),len(str(login_operation_counts)),len(str(login_daily_counts)),len(str(error_login_daily_counts)),len(str(error_login_counts.to_dict()))

(706, 1146, 5806, 2169, 2319, 138, 17)

In [29]:
daily_counts

日期
2025-03-01     52
2025-03-02      5
2025-03-03    114
2025-03-04     70
2025-03-05     78
             ... 
2025-08-27     26
2025-08-28     45
2025-08-29     84
2025-08-30      2
2025-08-31      3
Length: 184, dtype: int64

In [63]:
all_items = []
for date, inner_dict in login_operation_counts.items():
    for key, value in inner_dict.items():
        all_items.append((date, key, value))

# 按值降序排序
all_items.sort(key=lambda x: x[2], reverse=True)

# 获取前20个最大值
login_operation_counts = all_items[:20]


In [64]:
login_operation_counts

[('日指标“热点指标”数据查询', 'nc_gouxiang', 6185),
 ('日指标“猜你想看”数据查询', 'nc_gouxiang', 6181),
 ('日指标“热点指标”数据查询', 'zhenghan', 4489),
 ('日指标“猜你想看”数据查询', 'zhenghan', 4476),
 ('日指标“热点指标”数据查询', 'liurong68', 3399),
 ('日指标“猜你想看”数据查询', 'liurong68', 3397),
 ('日指标“猜你想看”数据查询', 'liuyu35', 2923),
 ('日指标“热点指标”数据查询', 'liuyu35', 2922),
 ('日指标“热点指标”数据查询', 'chengyuqiang', 2854),
 ('日指标“猜你想看”数据查询', 'chengyuqiang', 2854),
 ('日指标“热点指标”数据查询', 'yinjunshu', 2280),
 ('日指标“猜你想看”数据查询', 'yinjunshu', 2280),
 ('日指标“热点指标”数据查询', 'zhenzheng', 1736),
 ('月指标“猜你想看”数据查询', 'zhenghan', 1736),
 ('月指标“热点指标”数据查询', 'zhenghan', 1734),
 ('日指标“猜你想看”数据查询', 'zhenzheng', 1732),
 ('日指标“热点指标”数据查询', 'zhengjian1', 1686),
 ('日指标“猜你想看”数据查询', 'zhengjian1', 1686),
 ('日指标“热点指标”数据查询', 'helixing', 1552),
 ('日指标“猜你想看”数据查询', 'helixing', 1550)]

In [39]:
login_daily_counts = {
    date: {k: v for k, v in inner_dict.items() if v >200}
    for date, inner_dict in login_daily_counts.items()
}

In [46]:
error_login_daily_counts

{datetime.date(2025, 3, 10): {'chenxuelian12': 0,
  'chenyifan1': 0,
  'dfgx_wangshuping': 0,
  'dfgx_yuanhui': 0,
  'hanling2': 0,
  'hebiao1': 0,
  'limenghan': 0,
  'lipan13': 0,
  'liufei4': 0,
  'nc_gouxiang': 0,
  'nc_guolei': 0,
  'nc_liangbenyi': 0,
  'nc_wangzongwen': 0,
  'wuhongyu8': 0,
  'xudan25': 0,
  'yangceli': 0,
  'yukai': 0,
  'zhangyi17': 6,
  'zhouhong5': 0,
  'zhouquanwei': 0,
  'zhouyu': 0},
 datetime.date(2025, 3, 14): {'chenxuelian12': 4,
  'chenyifan1': 0,
  'dfgx_wangshuping': 0,
  'dfgx_yuanhui': 0,
  'hanling2': 0,
  'hebiao1': 0,
  'limenghan': 0,
  'lipan13': 0,
  'liufei4': 0,
  'nc_gouxiang': 0,
  'nc_guolei': 0,
  'nc_liangbenyi': 0,
  'nc_wangzongwen': 0,
  'wuhongyu8': 0,
  'xudan25': 0,
  'yangceli': 0,
  'yukai': 0,
  'zhangyi17': 0,
  'zhouhong5': 0,
  'zhouquanwei': 0,
  'zhouyu': 0},
 datetime.date(2025, 3, 21): {'chenxuelian12': 0,
  'chenyifan1': 0,
  'dfgx_wangshuping': 2,
  'dfgx_yuanhui': 10,
  'hanling2': 0,
  'hebiao1': 0,
  'limenghan': 

In [30]:
total_sum = 0
for inner_dict in login_daily_counts.values():
    for value in inner_dict.values():
        total_sum += value

In [31]:
for date, inner_dict in login_daily_counts.items():
    for key in inner_dict:
        inner_dict[key] = inner_dict[key] / total_sum

In [76]:
error_login_daily_counts

{datetime.date(2025, 3, 10): {'zhangyi17': 6},
 datetime.date(2025, 3, 14): {'chenxuelian12': 4},
 datetime.date(2025, 3, 21): {'dfgx_wangshuping': 2, 'dfgx_yuanhui': 10},
 datetime.date(2025, 4, 10): {'lipan13': 2},
 datetime.date(2025, 4, 19): {'limenghan': 12},
 datetime.date(2025, 4, 20): {'xudan25': 2},
 datetime.date(2025, 5, 1): {'nc_liangbenyi': 40},
 datetime.date(2025, 5, 6): {'yangceli': 2},
 datetime.date(2025, 5, 14): {'chenyifan1': 2},
 datetime.date(2025, 5, 18): {'liufei4': 2},
 datetime.date(2025, 6, 15): {'nc_guolei': 2},
 datetime.date(2025, 6, 17): {'zhouyu': 2},
 datetime.date(2025, 6, 27): {'liufei4': 2},
 datetime.date(2025, 7, 16): {'zhouquanwei': 16},
 datetime.date(2025, 7, 25): {'limenghan': 6},
 datetime.date(2025, 7, 27): {'hebiao1': 78},
 datetime.date(2025, 7, 28): {'yukai': 2},
 datetime.date(2025, 7, 30): {'nc_wangzongwen': 2},
 datetime.date(2025, 8, 2): {'nc_gouxiang': 4},
 datetime.date(2025, 8, 3): {'hanling2': 220},
 datetime.date(2025, 8, 31): {'w

In [42]:
import datetime
# 提取所有键值对，包含日期信息
all_items = []
for date, inner_dict in login_daily_counts.items():
    for key, value in inner_dict.items():
        all_items.append((date, key, value))

# 按值降序排序
all_items.sort(key=lambda x: x[2], reverse=True)

# 获取前20个最大值
top20 = all_items[:20]

# 打印结果
print("值最大的前20项：")
for i, (date, key, value) in enumerate(top20, 1):
    print(f"{i}. 日期: {date}, 帐号: {key}, 次数: {value}")


值最大的前20项：
1. 日期: 2025-08-07, 帐号: zhenghan, 次数: 846
2. 日期: 2025-05-22, 帐号: liufei4, 次数: 708
3. 日期: 2025-04-22, 帐号: nc_weiwei, 次数: 706
4. 日期: 2025-07-02, 帐号: zhenghan, 次数: 516
5. 日期: 2025-08-08, 帐号: zhenghan, 次数: 510
6. 日期: 2025-08-22, 帐号: zhenghan, 次数: 509
7. 日期: 2025-08-06, 帐号: zhenghan, 次数: 507
8. 日期: 2025-05-12, 帐号: zhenghan, 次数: 504
9. 日期: 2025-04-08, 帐号: chengyuqiang, 次数: 498
10. 日期: 2025-09-03, 帐号: nc_gouxiang, 次数: 495
11. 日期: 2025-03-03, 帐号: chengyuqiang, 次数: 480
12. 日期: 2025-05-06, 帐号: zhengjian1, 次数: 436
13. 日期: 2025-08-05, 帐号: nc_gouxiang, 次数: 430
14. 日期: 2025-04-02, 帐号: nc_houdan, 次数: 416
15. 日期: 2025-03-10, 帐号: chengyuqiang, 次数: 402
16. 日期: 2025-05-07, 帐号: zhenghan, 次数: 402
17. 日期: 2025-08-01, 帐号: zhenghan, 次数: 389
18. 日期: 2025-04-14, 帐号: yukai, 次数: 384
19. 日期: 2025-08-12, 帐号: nc_gouxiang, 次数: 368
20. 日期: 2025-04-03, 帐号: chengyuqiang, 次数: 362


In [36]:
# 获取前20个最大值
top20 = all_items[:20]

# 打印结果
print("值最大的前20项：")
for i, (date, key, value) in enumerate(top20, 1):
    print(f"{i}. 日期: {date}, 帐号: {key}, 次数: {value}")

值最大的前20项：
1. 日期: 2025-09-03, 帐号: nc_gouxiang, 次数: 495
2. 日期: 2025-09-04, 帐号: zhengjian1, 次数: 350
3. 日期: 2025-09-01, 帐号: zhenghan, 次数: 316
4. 日期: 2025-09-03, 帐号: zhengjian1, 次数: 230
5. 日期: 2025-09-04, 帐号: xszx_wangyue, 次数: 214
6. 日期: 2025-08-28, 帐号: nc_gouxiang, 次数: 190
7. 日期: 2025-08-31, 帐号: nc_gouxiang, 次数: 190
8. 日期: 2025-08-27, 帐号: nc_gouxiang, 次数: 188
9. 日期: 2025-09-01, 帐号: nc_gouxiang, 次数: 156
10. 日期: 2025-09-02, 帐号: nc_gouxiang, 次数: 156
11. 日期: 2025-09-04, 帐号: zhenghan, 次数: 148
12. 日期: 2025-09-04, 帐号: hanling2, 次数: 122
13. 日期: 2025-08-29, 帐号: nc_gouxiang, 次数: 120
14. 日期: 2025-08-25, 帐号: zhangweiwei2, 次数: 117
15. 日期: 2025-09-04, 帐号: nc_gouxiang, 次数: 116
16. 日期: 2025-09-04, 帐号: helixing, 次数: 115
17. 日期: 2025-08-30, 帐号: nc_gouxiang, 次数: 110
18. 日期: 2025-08-26, 帐号: nc_gouxiang, 次数: 108
19. 日期: 2025-09-03, 帐号: zhenghan, 次数: 102
20. 日期: 2025-09-02, 帐号: zhenghan, 次数: 94


In [1]:
! pip install pandoc
import pypandoc

def md_to_docx(md_file_path, docx_file_path):
    """
    将Markdown文件转换为Docx文件
    :param md_file_path: Markdown文件路径
    :param docx_file_path: 输出的Docx文件路径
    """
    try:
        # 进行格式转换
        output = pypandoc.convert_file(
            md_file_path,
            'docx',
            outputfile=docx_file_path
        )
        print(f"转换成功，文件已保存至：{docx_file_path}")
    except Exception as e:
        print(f"转换失败：{str(e)}")

# 使用示例
if __name__ == "__main__":
    # 输入的Markdown文件路径
    md_path = "日志分析报告.md"
    # 输出的Docx文件路径
    docx_path = "日志分析报告.docx"
    # 执行转换
    md_to_docx(md_path, docx_path)

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
转换失败：No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with included pandoc.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import pandas as pd
##登录日志
log_data =pd.read_excel(r"D:\work\中台\四川公司检查3个能力\中台检查文档需求清单-联邦特征工程组件库能力-0908\系统安全\1-1 日志查验\4A登录日志-联邦特征工程组件库能力.xlsx")

# 提取登录ID、登录时间、登出时间和客户端浏览器
log_data = log_data[['登录ID', '登录时间']]

# 确保登录时间和登出时间格式正确
log_data['登录时间'] = pd.to_datetime(log_data['登录时间'], format='%Y%m%d%H%M%S', errors='coerce')
log_data = log_data.dropna(subset=['登录时间'])
# 数据可视化：登录ID登录频率图
login_counts = log_data['登录ID'].value_counts().to_dict()

# 按登录时间分类统计
log_data['日期'] = log_data['登录时间'].dt.date
daily_counts = log_data.groupby('日期').size().to_dict()

# 每个登录ID在一天之内的登录次数排序
log_data = log_data[['登录ID', '登录时间']]

# 确保登录时间格式正确
log_data['登录时间'] = pd.to_datetime(log_data['登录时间'], format='%Y%m%d%H%M%S', errors='coerce')
log_data = log_data.dropna(subset=['登录时间'])

# 提取日期部分
log_data['日期'] = log_data['登录时间'].dt.date

# 按登录ID和日期分组，统计每个ID在每一天的登录次数
daily_login_counts = log_data.groupby(['登录ID', '日期']).size().reset_index(name='登录次数')

# 按登录次数降序排序
sorted_daily_login_counts = daily_login_counts.sort_values(by='登录次数', ascending=False)
pivot_table = sorted_daily_login_counts[:30].pivot(index='登录ID', columns='日期', values='登录次数').fillna(0).to_dict()
pivot_table = {
    date: {k: v for k, v in inner_dict.items() if v != 0}
    for date, inner_dict in pivot_table.items()
}

error_time = log_data[(log_data['登录时间'].dt.hour >= 0) & (log_data['登录时间'].dt.hour < 6)]
error_daily_login_counts = error_time.groupby(['登录ID', '日期']).size().reset_index(name='登录次数')
error_sorted_daily_login_counts = error_daily_login_counts.sort_values(by='登录次数', ascending=False)
error_pivot_table = error_sorted_daily_login_counts[:30].pivot(index='登录ID', columns='日期',
                                                               values='登录次数').fillna(0).to_dict()

error_login_counts = error_time['登录ID'].value_counts().to_dict()

In [31]:
print("login_counts", len(str(login_counts)))
print("daily_counts", len((str(daily_counts))))
print("pivot_table", len((str(pivot_table))))
print("error_login_counts", len((str(error_login_counts))))
print("error_pivot_table", len((str(error_pivot_table))))

login_counts 16
daily_counts 614
pivot_table 894
error_login_counts 2
error_pivot_table 2


In [32]:
pivot_table

{datetime.date(2025, 3, 31): {'xuxiao17': 4},
 datetime.date(2025, 4, 7): {'xuxiao17': 2},
 datetime.date(2025, 4, 8): {'xuxiao17': 2},
 datetime.date(2025, 4, 9): {'xuxiao17': 3},
 datetime.date(2025, 4, 10): {'xuxiao17': 2},
 datetime.date(2025, 4, 11): {'xuxiao17': 1},
 datetime.date(2025, 4, 25): {'xuxiao17': 1},
 datetime.date(2025, 4, 27): {'xuxiao17': 2},
 datetime.date(2025, 5, 9): {'xuxiao17': 1},
 datetime.date(2025, 5, 12): {'xuxiao17': 1},
 datetime.date(2025, 5, 15): {'xuxiao17': 3},
 datetime.date(2025, 5, 30): {'xuxiao17': 1},
 datetime.date(2025, 6, 5): {'xuxiao17': 1},
 datetime.date(2025, 6, 17): {'xuxiao17': 2},
 datetime.date(2025, 6, 18): {'xuxiao17': 2},
 datetime.date(2025, 6, 23): {'xuxiao17': 2},
 datetime.date(2025, 6, 30): {'xuxiao17': 2},
 datetime.date(2025, 7, 9): {'xuxiao17': 2},
 datetime.date(2025, 7, 31): {'xuxiao17': 3},
 datetime.date(2025, 8, 19): {'xuxiao17': 2}}

In [ ]:
error_login_daily_counts = error_time.groupby(['登录ID', '日期']).size().unstack(fill_value=0).to_dict()
    error_login_daily_counts = {
        date: {k: v for k, v in inner_dict.items() if v != 0}
        for date, inner_dict in error_login_daily_counts.items()
    }

In [40]:
# 假设日志数据存储在CSV文件中
import pandas as pd
# 提取主帐号名称、操作时间和操作内容
##操作日志
log_data =pd.read_excel(r"D:\work\中台\四川公司检查3个能力\中台检查文档需求清单-联邦特征工程组件库能力-0908\系统安全\1-1 日志查验\主机操作日志-联邦特征工程组件库能力.xlsx")

log_data = log_data[['主帐号名称', '操作时间', '操作内容']]

# 确保操作时间格式正确
log_data['操作时间'] = pd.to_datetime(log_data['操作时间'], errors='coerce')
log_data = log_data.dropna(subset=['操作时间'])

# 按主帐号名称分类统计
account_counts = log_data['主帐号名称'].value_counts().to_dict()

log_data['日期'] = log_data['操作时间'].dt.date
daily_counts = log_data.groupby('日期').size().to_dict()

# 按操作内容分类统计
operation_counts = log_data['操作内容'].value_counts().to_dict()

# 多维度组合分析：按主账号和操作时间分组
account_daily_counts = log_data.groupby(['主帐号名称', '日期']).size().unstack(fill_value=0).to_dict()

error_time = log_data[((log_data['操作时间'].dt.hour >= 0) & (log_data['操作时间'].dt.hour < 6))]

error_login_daily_counts = error_time.groupby(['主帐号名称', '日期']).size().unstack(fill_value=0).to_dict()
error_login_daily_counts = {
    date: {k: v for k, v in inner_dict.items() if v != 0}
    for date, inner_dict in error_login_daily_counts.items()
}
error_login_counts = error_time['主帐号名称'].value_counts().to_dict()

In [41]:
account_counts

主帐号名称
xuxiao17    49
Name: count, dtype: int64

In [42]:
daily_counts

日期
2025-04-07    19
2025-04-10    30
dtype: int64

In [43]:
operation_counts

操作内容
ls                               12
ll                                9
cd ..                             4
cd sftp/                          2
lsblk                             2
df -h                             2
cd /data                          2
pwd                               2
cd beuser/                        1
cd sftp/be/                       1
cd gaia                           1
cd gaia_bak/                      1
cd /                              1
cd smoke_testing/                 1
docker ps -a                      1
cat /etc/fstab                    1
cd /gaia/sftp/be/                 1
cd be/                            1
docker logs -f cb6                1
cd /gaia                          1
cd sftp/beuser/smoke_testing/     1
mount -a                          1
Name: count, dtype: int64

In [44]:
account_daily_counts

日期,2025-04-07,2025-04-10
主帐号名称,,
xuxiao17,19,30


In [45]:
error_login_counts

Series([], Name: count, dtype: int64)

In [46]:
error_login_daily_counts

{}